# 03 构建最终九类因子

因子定义、输入校验和特征函数集中在 `src/treasury_futures/factors/cicc_close_session_reverse/build.py`；本 Notebook 只执行构建与导出。


In [1]:
from pathlib import Path
import sys

_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for candidate in _PROJECT_CANDIDATES if (candidate / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("找不到项目根目录 pyproject.toml；请从本项目内启动 Jupyter。")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from treasury_futures.factors.cicc_close_session_reverse.build import *  # noqa: F403


## 导出九个语义因子

只输出语义 factor_id、中文名称、类别、持有天数、方向与信号，不输出研究阶段旧规则别名。

In [2]:
daily_df, minute_df, events_df = load_inputs()
minute_ready = prepare_minute(minute_df)
features = build_segment_features(minute_ready)
features = add_reversal_and_v2(features)
features = add_oi_trend_event_filters(features, daily_df, events_df)
features = add_semantic_triggers(features)
threshold_audit = assert_historical_thresholds(features, daily_df)
signal_columns = CATALOG["factor_id"].tolist()
signals = features.melt(id_vars=["trading_date"], value_vars=signal_columns, var_name="factor_id", value_name="signal")
signals = signals[signals["signal"].fillna(False)].copy()
signals = signals.rename(columns={"trading_date": "signal_date"})
signals = signals.merge(CATALOG, on="factor_id", how="left")
signals["signal_date"] = pd.to_datetime(signals["signal_date"]).dt.normalize()

# 将逐交易日移仓监控结果拼到每一条信号上；监控表每个 trade_date 只能有一行。
roll_monitor = load_roll_monitor()
signals = signals.merge(
    roll_monitor,
    left_on="signal_date",
    right_on="trade_date",
    how="left",
    validate="many_to_one",
)

# 找不到监控日期时采用 fail-safe：保留信号行，但标记为必须屏蔽。
missing_monitor_mask = signals["trade_date"].isna()
missing_monitor_dates = int(signals.loc[missing_monitor_mask, "signal_date"].nunique())
signals["roll_status"] = signals["roll_status"].fillna("NO_MONITOR_DATA")
signals["block_signal"] = signals["block_signal"].astype("boolean").fillna(True).astype(bool)
blocked_signal_rows = int(signals["block_signal"].sum())
blocked_signal_dates = int(signals.loc[signals["block_signal"], "signal_date"].nunique())

# 这里只追加监控字段，不在构建阶段删除信号；是否过滤由后续回测决定。
signals = signals[["signal_date", "trade_date", "roll_status", "block_signal", "factor_id", "display_name", "category", "hold_days", "direction", "signal"]].sort_values(["signal_date", "factor_id"]).reset_index(drop=True)
quality = pd.DataFrame([
    {"check": "validated_inputs", "status": "PASS", "detail": f"daily={len(daily_df)}, minute={len(minute_df)}, events={len(events_df)}"},
    {"check": "catalog_count", "status": "PASS", "detail": f"exactly {len(CATALOG)} semantic factors"},
    {"check": "historical_thresholds", "status": "PASS", "detail": f"{int(threshold_audit['rows_checked'].sum())} threshold rows checked with source date < signal date"},
    {"check": "roll_monitor_merge", "status": "PASS" if missing_monitor_dates == 0 else "WARN", "detail": f"monitor_rows={len(roll_monitor)}, missing_signal_dates={missing_monitor_dates}, blocked_signal_rows={blocked_signal_rows}, blocked_signal_dates={blocked_signal_dates}"},
    {"check": "signal_rows", "status": "PASS", "detail": f"{len(signals)} triggered signal rows"},
])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(SIGNAL_XLSX, engine="openpyxl") as writer:
    signals.to_excel(writer, sheet_name="signals", index=False)
    CATALOG.to_excel(writer, sheet_name="factor_catalog", index=False)
    quality.to_excel(writer, sheet_name="build_quality", index=False)
print(f"已输出 {SIGNAL_XLSX}，信号行数={len(signals)}，因子数={len(CATALOG)}")
print(f"移仓监控：屏蔽信号行数={blocked_signal_rows}，涉及交易日={blocked_signal_dates}，缺失监控日期={missing_monitor_dates}")

VWAP 成交额单位识别: multiplier=10,000, median_ratio=10,000.000
已输出 E:\CodeBase\TaiKangAM\TreasuryFutures\Factor\notebooks\CICC_CloseSession_Reverse\CTreasuryFutures\factors\cicc_close_session_reverse\output\final_9_category_signals.xlsx，信号行数=126，因子数=9
移仓监控：屏蔽信号行数=32，涉及交易日=15，缺失监控日期=0
